# Guide d'utilisation du plugin de backtest factoriel

Ce notebook montre comment appeler `func.py` et `BacktestEngine.py` depuis le même dossier. Le flux principal est le suivant : préparer les données → tester les variables brutes → tester les signaux unitaires dérivés → effectuer une analyse incrémentale → construire un score composite large.

Chaque backtest peut retourner une figure Plotly. Lors de l'export centralisé, les sorties HTML et PNG peuvent être activées séparément. Les figures présentent Top, Worst et Benchmark, ainsi que les ratios Top/Bench, Worst/Bench et Top/Worst.

## Trois points d'entrée principaux

- `test_raw_variables()` : teste directement une liste de colonnes existantes, sans direction ni configuration de signal. Top correspond aux valeurs brutes les plus élevées et Worst aux valeurs les plus faibles.
- `test_unitary_signals()` : teste séparément level, pct et diff pour chaque variable. Cette fonction utilise `higher_is_better` et `denominator`, mais ignore `use_*` et `weight_*`.
- `test_incremental_signals()` : ajoute chaque variable candidate séparément à un score de base. Les dimensions et les poids réellement utilisés proviennent de `candidate_config`.

`calculate_composite_score()` construit le score composite en respectant strictement `use_level/use_pct/use_diff` et leurs poids, puis conserve les colonnes dérivées dans `screen`.

## 1. Charger le plugin

In [ ]:
from pathlib import Path
import importlib
import sys
import pandas as pd

PLUGIN_DIR = Path(r"C:\Users\jingx\Downloads\公司回测插件")
if str(PLUGIN_DIR) not in sys.path:
    sys.path.insert(0, str(PLUGIN_DIR))

import BacktestEngine
import func

importlib.reload(BacktestEngine)
importlib.reload(func)

from func import (
    RECOMMENDED_PERIOD_BREAKPOINTS,
    build_periods_from_breakpoints,
    calculate_composite_score,
    calculate_performance_ratios,
    combine_backtest_performances,
    compare_backtest_results,
    export_backtest_results,
    plot_performance_comparison,
    reconstruct_backtest_analysis,
    run_top_worst_backtest,
    test_composite_signal,
    test_incremental_signals,
    test_raw_variables,
    test_unitary_signals,
)

print("Plugin chargé")

## 2. Préparer les données

Avant d'exécuter les cellules suivantes, il faut préparer :

- `screen` : DataFrame transversal des actions contenant au minimum `ISIN`, `Date`, le secteur, le pays et les variables à tester. Il doit également contenir les colonnes requises par `PtfBuilder`, notamment le SEDOL, la capitalisation et les poids du benchmark.
- `returns` : DataFrame des rendements au format attendu par `BacktestEngine`.
- `list_noire_path` : chemin du fichier Excel de liste noire. Dans cet exemple, il est fixé à `None`.

La cellule suivante charge directement les fichiers parquet réels du dossier `data` du plugin.

In [ ]:
DATA_DIR = PLUGIN_DIR / "data"
SCREEN_PATH = DATA_DIR / "screen_aggregate.parquet"
RETURNS_PATH = DATA_DIR / "returns.parquet"

screen = pd.read_parquet(SCREEN_PATH)
returns = pd.read_parquet(RETURNS_PATH)
list_noire_path = None

In [ ]:
missing_objects = [name for name in ("screen", "returns", "list_noire_path") if name not in globals()]
if missing_objects:
    raise RuntimeError(f"Préparez d'abord ces objets : {missing_objects}")

required_columns = {
    "Date",
    " Benchmark ICB Supersector ",
    "Exchange Country Region",
}
missing_columns = sorted(required_columns - set(screen.columns))
if "ISIN" not in screen.columns and screen.index.name != "ISIN":
    missing_columns.append("ISIN")
if missing_columns:
    raise KeyError(f"Colonnes de base absentes de screen : {missing_columns}")

screen["Date"] = pd.to_datetime(screen["Date"])
print(f"screen : {screen.shape}, returns : {returns.shape}")

## 3. Définir la configuration des signaux dans le notebook

La configuration ci-dessous est uniquement un exemple de format. Ajoutez ou supprimez les variables en fonction des colonnes réellement présentes dans `screen`.

Règles de configuration :

- `higher_is_better` : indique si une valeur élevée représente une meilleure direction.
- `denominator` : divise d'abord une variable absolue par cette colonne, par exemple R&D / Sales.
- `use_level/use_pct/use_diff` : dimensions réellement utilisées par le score composite et l'analyse incrémentale.
- `weight_*` : poids de chaque dimension dans le score composite.

In [ ]:
WIDE_CONFIG = {
    "Revenue 5Y CAGR": {
        "higher_is_better": True,
        "use_level": True, "weight_level": 1.0,
        "use_pct": True, "weight_pct": 1.0,
        "use_diff": False, "weight_diff": 0.0,
    },
    "FCF Conversion": {
        "higher_is_better": True,
        "use_level": True, "weight_level": 1.0,
        "use_pct": False, "weight_pct": 0.0,
        "use_diff": True, "weight_diff": 1.0,
    },
    "Net Debt to Ebit": {
        "higher_is_better": False,
        "use_level": True, "weight_level": 1.0,
        "use_pct": False, "weight_pct": 0.0,
        "use_diff": True, "weight_diff": 1.0,
    },
    "R&D Expense CIQ": {
        "higher_is_better": True,
        "denominator": "Sales",
        "use_level": True, "weight_level": 1.0,
        "use_pct": False, "weight_pct": 0.0,
        "use_diff": False, "weight_diff": 0.0,
    },
}

available_config = {name: options for name, options in WIDE_CONFIG.items() if name in screen.columns}
missing_config = sorted(set(WIDE_CONFIG) - set(available_config))
print(f"Variables configurées disponibles : {list(available_config)}")
print(f"Variables absentes de screen et ignorées : {missing_config}")

## 4. Utilisation la plus simple : tester directement les variables brutes

Cette étape ne nécessite aucune configuration de signal. Chaque variable produit une comparaison Top/Worst/Benchmark. Top représente toujours le groupe ayant les valeurs brutes les plus élevées ; cela ne signifie pas nécessairement que ce groupe est économiquement meilleur.

In [ ]:
RAW_VARIABLES = [
    "Revenue 5Y CAGR",
    "FCF Conversion",
    "Net Debt to Ebit",
]

raw_results = test_raw_variables(
    screen=screen,
    returns=returns,
    variables=RAW_VARIABLES,
    list_noire_path=list_noire_path,
    percentile=0.13,
    show_plot=True,
)

In [ ]:
raw_summary = pd.DataFrame([
    {
        "Variable": name,
        "Robust Score": result.get("robust_score"),
        "Top/Bench": result.get("top_bench_ratio"),
        "Top/Worst": result.get("top_worst_ratio"),
    }
    for name, result in raw_results.items()
]).sort_values("Robust Score", ascending=False)
raw_summary

## 5. Tester les signaux unitaires dérivés

Par défaut, chaque variable est testée séparément en level, pct et diff ; une variable peut donc produire jusqu'à trois figures. Cette fonction ignore les paramètres `use_*` et les poids de la configuration, mais utilise la direction et le dénominateur.

Pour ne tester que certaines dimensions, utilisez par exemple `dimensions=("pct",)` ou `dimensions=("level", "diff")`.

In [ ]:
unitary_results = test_unitary_signals(
    screen=screen,
    returns=returns,
    signal_config=available_config,
    list_noire_path=list_noire_path,
    dimensions=("level", "pct", "diff"),
    percentile=0.13,
    show_plot=True,
)

In [ ]:
derived_columns = [
    column for column in screen.columns
    if "__over__" in column or column.endswith("__pct") or column.endswith("__diff")
]
derived_columns

Les colonnes dérivées restent dans `screen`. Elles peuvent donc être testées ensuite comme de simples variables brutes.

In [ ]:
derived_to_test = derived_columns[:5]
derived_raw_results = test_raw_variables(
    screen=screen,
    returns=returns,
    variables=derived_to_test,
    list_noire_path=list_noire_path,
    percentile=0.13,
    show_plot=True,
) if derived_to_test else {}

## 6. Effectuer une analyse incrémentale

Le score de base est d'abord backtesté une fois. Chaque variable candidate est ensuite ajoutée séparément à cette base. Les dimensions level, pct et diff ainsi que leurs poids proviennent entièrement de `CANDIDATE_CONFIG`.

In [ ]:
BASELINE_CONFIG = {
    "Quality Avg Percentile": {
        "higher_is_better": True,
        "use_level": True, "weight_level": 1.0,
        "use_pct": False, "weight_pct": 0.0,
        "use_diff": False, "weight_diff": 0.0,
    },
}

CANDIDATE_CONFIG = available_config

incremental_results = test_incremental_signals(
    screen=screen,
    returns=returns,
    baseline_config=BASELINE_CONFIG,
    candidate_config=CANDIDATE_CONFIG,
    list_noire_path=list_noire_path,
    percentile=0.13,
    show_plot=True,
)

In [ ]:
incremental_summary = pd.DataFrame([
    {
        "Test": name,
        "Robust Score": payload["backtest"].get("robust_score"),
        "Top/Bench": payload["backtest"].get("top_bench_ratio"),
        "Top/Worst": payload["backtest"].get("top_worst_ratio"),
    }
    for name, payload in incremental_results.items()
]).sort_values("Robust Score", ascending=False)
incremental_summary

## 7. Construire et backtester un score composite large

Cette étape utilise strictement les dimensions activées et les poids de `WIDE_CONFIG`. Le score `Score_Wide` et les colonnes dérivées associées sont ajoutés directement à `screen`. Le résultat fournit aussi `raw_variables`, `raw_variable_weights` et la composition détaillée par dimension.

In [ ]:
wide_results = test_composite_signal(
    screen=screen,
    returns=returns,
    score_col="Score_Wide",
    signal_config=available_config,
    list_noire_path=list_noire_path,
    percentile=0.13,
    show_plot=True,
)
screen = wide_results["screen"]
wide_result = wide_results["backtest"]

In [ ]:
display(pd.Series({
    "Robust Score": wide_result.get("robust_score"),
    "Top/Bench": wide_result.get("top_bench_ratio"),
    "Top/Worst": wide_result.get("top_worst_ratio"),
}, name="Score_Wide"))

print("Variables brutes :", wide_results["raw_variables"])
display(pd.DataFrame.from_dict(
    wide_results["raw_variable_weights"], orient="index"
))
display(wide_results["composition"])

## 8. Comparer et exporter tous les résultats

Placez les différents types de résultats dans un même dictionnaire. `compare_backtest_results()` produit une table triable qui présente également les variables brutes, les dimensions, les directions et la recette des poids de chaque test.

Il est conseillé de comparer d'abord `robust_rank_within_type` au sein d'une même famille de tests, puis d'examiner les composantes du Robust Score et enfin les courbes Top/Bench et Top/Worst. Pour les candidats incrémentaux, les colonnes `*_delta_vs_baseline` indiquent directement l'amélioration par rapport à la base.

In [ ]:
ALL_RESULTS = {
    "raw": raw_results,
    "unitary": unitary_results,
    "derived_raw": derived_raw_results,
    "incremental": incremental_results,
    "wide": {"Score_Wide": wide_results},
}

comparison_table = compare_backtest_results(ALL_RESULTS)
comparison_table

In [ ]:
classic_view = comparison_table[[
    "test_type", "test_name", "robust_score",
    "top_total_return", "top_annualized_return",
    "top_annualized_volatility", "top_sharpe_ratio",
    "top_max_drawdown", "top_information_ratio",
    "top_beta", "top_tracking_error", "top_sortino_ratio",
]].sort_values("robust_score", ascending=False)
classic_view

Le dossier d'export contient :

- `backtest_summary.csv` : toutes les métriques, les classements, les variables brutes, le résumé des poids et la recette lisible ;
- `signal_composition.csv` : une ligne par variable et dimension, avec le poids total de la variable brute et sa part du poids absolu ;
- `classic_metrics.csv` : les métriques classiques de Top, Worst et Benchmark au format long ;
- `period_metrics.csv` : les CAGR, les métriques classiques et les classements de chaque test pour chaque période ;
- `metrics_by_period.csv` : la table unifiée test × période qui réunit la période totale et toutes les sous-périodes ;
- `backtest_registry.json` : le registre complet lisible par machine ;
- `figures/` : les figures HTML ou PNG selon `export_html` et `export_png` ;
- `data/` : les courbes de valeur et les ratios de chaque test ;
- `holdings/` : les positions historiques Top et Worst de chaque test.

Tous les fichiers de données et toutes les tables récapitulatives sont écrits avant les images. L'export PNG dépend de Kaleido. Si les images ne sont pas nécessaires, désactivez à la fois `export_html` et `export_png`.

In [ ]:
# Décommentez cette ligne une seule fois si l'export PNG est indisponible.
# %pip install --upgrade kaleido

In [ ]:
export_bundle = export_backtest_results(
    results=ALL_RESULTS,
    output_dir=PLUGIN_DIR / "exports",
    export_name="factor_research_run",
    export_html=False,
    export_png=False,
)

export_bundle["export_dir"]

### Reconstruire et combiner les performances

`combine_backtest_performances()` utilise en priorité `ALL_RESULTS` en mémoire et complète les résultats absents avec les fichiers `data/*_performance.csv` du dossier exporté. Après un redémarrage du kernel, `globals().get("ALL_RESULTS")` renvoie `None` et la fonction restaure automatiquement toutes les performances depuis le disque.

`reconstruct_backtest_analysis()` restaure en une seule fois les performances, les compositions, tous les scores et métriques de la période totale, la table longue des métriques classiques et toutes les métriques par sous-période. `metrics_by_period` réunit `Période totale` et chaque période de rupture dans une seule table test × période, adaptée aux filtres et tableaux croisés. Après un redémarrage du kernel, les tables CSV officielles sont rechargées automatiquement ; si les résultats existent en mémoire, elles sont reconstruites directement. Le libellé automatique d'un composite présente ses variables brutes, les dimensions activées et leurs poids, par exemple `composite / Revenue 5Y CAGR[level×1.0, pct×0.5] | FCF Conversion[level×1.0, diff×0.5] | Net Debt to Ebit[level×1.0, diff×0.5] | Top`. Les ratios sont calculés séparément par `calculate_performance_ratios()` ; `plot_performance_comparison()` se limite à tracer les deux panneaux Plotly.

In [ ]:
EXPORT_DIR = PLUGIN_DIR / "exports" / "demo_real_data_20260719"

analysis_bundle = reconstruct_backtest_analysis(
    results=globals().get("ALL_RESULTS"),
    export_dir=EXPORT_DIR,
    portfolios=("Top",),
    performance_save_path=EXPORT_DIR / "all_top_performance.csv",
)

all_top_performance = analysis_bundle["performance"]
performance_composition = analysis_bundle["performance_composition"]
backtest_summary = analysis_bundle["summary"]
classic_metrics = analysis_bundle["classic_metrics"]
period_metrics = analysis_bundle["period_metrics"]
metrics_by_period = analysis_bundle["metrics_by_period"]
signal_composition = analysis_bundle["signal_composition"]

composite_composition = performance_composition.loc[
    performance_composition["test_type"].eq("composite"),
    [
        "display_path", "raw_variable", "dimension",
        "higher_is_better", "weight", "denominator",
        "raw_variable_total_weight",
        "raw_variable_absolute_weight_share",
    ],
]

display(all_top_performance)
display(composite_composition)
display(backtest_summary)
display(classic_metrics)
display(metrics_by_period)

In [ ]:
COMPOSITE_TEST_PATH = performance_composition.loc[
    performance_composition["test_type"].eq("composite"), "test_path"
].drop_duplicates().iloc[0]

PERFORMANCE_SELECTION = {
    "Top Revenue brut": ("raw / Revenue 5Y CAGR", "Top"),
    "Top Revenue unitaire level": ("unitary / Revenue 5Y CAGR | level", "Top"),
    "Top FCF incrémental": ("incremental / FCF Conversion", "Top"),
    "Top composite": (COMPOSITE_TEST_PATH, "Top"),
    "Benchmark": (COMPOSITE_TEST_PATH, "Bench"),
}

selected_performance = combine_backtest_performances(
    results=globals().get("ALL_RESULTS"),
    export_dir=EXPORT_DIR,
    selections=PERFORMANCE_SELECTION,
    save_path=EXPORT_DIR / "selected_performance_comparison.csv",
)

selected_ratios = calculate_performance_ratios(
    selected_performance,
    benchmark_column="Benchmark",
)

comparison_figure = plot_performance_comparison(
    performance=selected_performance,
    ratios=selected_ratios,
    benchmark_column="Benchmark",
    title="Comparaison des performances",
    save_path=None,
    show_plot=True,
)

## 9. Comparer les différentes périodes

Tous les points d'entrée utilisent un seul paramètre temporel : la liste d'années `period_breakpoints`. La valeur recommandée `[2020, 2022, 2024]` découpe la chronologie en quatre segments : avant 2020, 2020–2021, 2022–2023 et depuis 2024. Le menu déroulant situé en haut de chaque figure Plotly affiche, pour la période choisie, le CAGR de Top, le CAGR actif et le CAGR Top/Worst.

`backtest_summary.csv` contient les principaux champs au format large `period_<période>_<métrique>`. `period_metrics.csv` contient une ligne complète par couple test × période et convient mieux aux filtres et aux tableaux croisés.

In [ ]:
pd.DataFrame(build_periods_from_breakpoints(RECOMMENDED_PERIOD_BREAKPOINTS))

In [ ]:
period_table = period_metrics
period_view = period_table[[
    "period_label", "test_type", "test_name",
    "top_cagr", "active_cagr", "top_worst_cagr",
    "top_sharpe_ratio", "top_max_drawdown",
    "active_cagr_rank_global", "active_cagr_rank_within_type",
]]
period_view.sort_values(["period_label", "active_cagr_rank_global"])

In [ ]:
active_cagr_pivot = period_table.pivot_table(
    index=["test_type", "test_name"],
    columns="period_id",
    values="active_cagr",
)
active_cagr_pivot.sort_values(active_cagr_pivot.columns[-1], ascending=False)

Une année de rupture marque le début d'une nouvelle période. Ainsi, `[2009, 2020]` crée trois segments : avant 2009, 2009–2019 et depuis 2020. Les ruptures recommandées dans le code sont 2020, 2022 et 2024. Ajoutez 2009 si les données couvrent aussi les années antérieures à 2009. Une liste vide `[]` conserve uniquement la période totale.

In [ ]:
PERIOD_BREAKPOINTS = [2009, 2020]
display(pd.DataFrame(build_periods_from_breakpoints(PERIOD_BREAKPOINTS)))

# Exemple d'appel direct avec les points de rupture.
# breakpoint_results = test_raw_variables(
#     screen, returns, RAW_VARIABLES, list_noire_path,
#     period_breakpoints=PERIOD_BREAKPOINTS,
# )

## 10. Lire les objets de résultats

Un résultat de backtest standard contient notamment :

```python
result["robust_score"]
result["top_bench_ratio"]
result["top_worst_ratio"]
result["performance"]
result["ratios"]
result["figure"]
result["top_holdings"]
result["worst_holdings"]
result["period_metrics"]
```

Les résultats incrémentaux ajoutent un niveau `backtest` :

```python
incremental_results["Net Debt to Ebit"]["backtest"]["robust_score"]
```

Le Robust Score nécessite au moins 756 jours de cotation communs. Si l'historique est insuffisant, il renvoie `NaN`.

Le flux interne est maintenant découplé : `calculate_top_vs_bottom_results()` produit d'abord les performances, les ratios et toutes les métriques, puis `plot_top_vs_bottom_results()` construit la figure à partir de cet objet. Pour un traitement en lot limité aux données, utilisez `show_plot=False, build_figure=False` dans n'importe quel point d'entrée ; les comparaisons restent reconstructibles depuis les fichiers performance locaux.